# Azure attempt to access Open Datasets

In [ ]:
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential

# ml_client = MLClient(DefaultAzureCredential(), subscription_id="182b54b3-398b-4419-a1e1-baa12cf196bf", resource_group_name="NTAB-MSAI", workspace_name="your-workspace")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


# MIND Finance Access (not useful)

In [1]:
import pandas as pd

In [2]:
raw_train_news_df = pd.read_csv("F:/NT@B/Microsoft-sp26/MINDlarge_train/news.tsv", sep="\t")
# raw_test_news_df = pd.read_csv("F:/NT@B/Microsoft-sp26/MINDlarge_test/news.tsv", sep="\t")

header_data = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
raw_train_news_df.columns = header_data
# raw_test_news_df.columns = header_data

train_df = raw_train_news_df
train_df.head()

,id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N45436,news,newsscienceandtechnology,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,https://assets.msn.com/labs/mind/AABmf2I.html,"[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...","[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ..."
1,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[]
3,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
4,N75236,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."


In [7]:
finance_df = train_df[train_df['category'] == 'finance']
finance_df = finance_df.dropna()
finance_df.head()

,id,category,subcategory,title,abstract,url,title_entities,abstract_entities
57,N55720,finance,finance-insurance,10 dental scams that can bite you hard,1,https://assets.msn.com/labs/mind/AACI1qe.html,[],[]
86,N35617,finance,finance-real-estate,The 25 most desirable places to live in the US...,Check out where U.S. residents would live if t...,https://assets.msn.com/labs/mind/AABvlID.html,[],"[{""Label"": ""United States"", ""Type"": ""G"", ""Wiki..."
93,N13286,finance,financenews,Low Income Seniors At Risk Of Homelessness In ...,More than 100 low income senior citizens livin...,https://assets.msn.com/labs/mind/AAJgOFd.html,"[{""Label"": ""Novato, California"", ""Type"": ""G"", ...","[{""Label"": ""Novato, California"", ""Type"": ""G"", ..."
119,N116074,finance,finance-insurance,3 ways Halloween can pose an insurance risk,Do you need Halloween insurance?,https://assets.msn.com/labs/mind/AAI3Dd9.html,"[{""Label"": ""Halloween"", ""Type"": ""H"", ""Wikidata...","[{""Label"": ""Halloween"", ""Type"": ""H"", ""Wikidata..."
128,N108765,finance,finance-career-education,"These VA, DC Universities Among Best In The Wo...",New rankings from U.S. News & World Report eva...,https://assets.msn.com/labs/mind/AAJaBSJ.html,"[{""Label"": ""U.S. News & World Report"", ""Type"":...","[{""Label"": ""U.S. News & World Report"", ""Type"":..."


In [13]:
finance_df['abstract'].iloc[2]

"More than 100 low income senior citizens living in affordable housing in Novato are worried they're being priced out."

# Manually scraped MSN Finance

In [1]:
with open("F:/NT@B/Microsoft-sp26/sample_finance_article.txt", "r") as f:
    manual_article = f.read()

In [2]:
manual_article

'Jensen Huang\'s vision of AI agents everywhere aligns perfectly with ServiceNow\'s core business model. ServiceNow\'s stock has been dragged down by the SaaS sell-off, but its plummet is undeserved. Investors are afraid of AI disrupting SaaS companies, but ServiceNow\'s platform enables AI business disruption. 10 stocks we like better than Nvidia â€º Years ago, E.F. Hutton ran a commercial that proclaimed, "When E.F. Hutton speaks, people listen." We could perhaps replace E.F. Hutton with Jensen Huang in that statement today. When the Nvidia (NASDAQ: NVDA) CEO speaks, people listen. And Huang spoke at length at his company\'s 2026 GTC AI conference last week.'

In [9]:
import os
import json
import math
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import torch
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from sklearn.model_selection import train_test_split

## Configuration

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./qwen2.5_finance_lora"

CUSTOM_ENTITY_AFFECT_JSONL = "entity_affect_train_normalized.jsonl"  # Path to custom entity affect dataset in JSONL format, or None to skip this dataset
CUSTOM_ENTITY_AFFECT_VAL_JSONL = "entity_affect_val_normalized.jsonl"  # Path to custom entity affect validation dataset in JSONL format, or None to skip this dataset

MAX_LENGTH = 512
SEED = 42

USE_4BIT = False
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16 = torch.cuda.is_available() and not BF16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_EPOCHS = 2
LR = 2e-4
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 20
SAVE_STEPS = 100
EVAL_STEPS = 100

DEBUG_MAX_TRAIN = 10  # Set to a small number for quick debugging, or None to use the full dataset
DEBUG_MAX_EVAL = 10  # Set to a small number for quick debugging, or None to use the full dataset

FINANCE_ENTITY_TYPES = ["Company", 
                        "FinancialInstrument", 
                        "EconomicIndicator", 
                        "MarketIndex", 
                        "Currency", 
                        "FinancialEvent", 
                        "Person", 
                        "Ticker", 
                        "Sector", 
                        "Asset", 
                        "Commodity", 
                        "Index", 
                        "Country_or_Region",
                        "GovernemntBody",
                        "CentralBank",
                        "FinancialInstitution",
                        "MacroIndicator",
                        "MicroIndicator",
                        "Policy_or_Regulation"]

SYSTEM_PROMPT = f"""You are a financial NLP system. Follow the user's task exactly. When asked to extract entities, use only these entity types: {FINANCE_ENTITY_TYPES}. 

For entity affect scoring:
- sentiment must be one of: positive, neutral, negative
- valence must be a float in [-1.0, 1.0]
- arousal must be a float in [0.0, 1.0]
- return strict JSON only when the user asks for JSON
"""

random.seed(SEED)

## Helpers

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def safe_json_dumps(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)


def build_chat_example(user_prompt: str, assistant_response: str) -> Dict[str, str]:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_response},
        ] # type: ignore
    }


def format_messages_as_text(messages: List[Dict[str, str]], tokenizer) -> str:
    """
    Uses tokenizer chat template if available.
    Falls back to a simple text format otherwise.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    # Fallback
    chunks = []
    for msg in messages:
        chunks.append(f"{msg['role'].upper()}: {msg['content']}")
    return "\n\n".join(chunks)

## Data Conversion

In [5]:
def convert_financial_phrasebank() -> Optional[Dataset]:
    """
    Expected label mapping:
      0 -> negative
      1 -> neutral
      2 -> positive
    You may need to adjust depending on the version you load.
    """
    try:
        ds = load_dataset("takala/financial_phrasebank", "sentences_allagree", trust_remote_code=True)
    except Exception as e:
        print(f"[WARN] Could not load FinancialPhraseBank: {e}")
        return None

    label_map = {
        0: "negative",
        1: "neutral",
        2: "positive",
    }

    def mapper(ex):
        text = ex["sentence"]
        label = label_map[int(ex["label"])]
        prompt = f"""Task: classify_financial_sentiment
            Text:
            {text}

            Return JSON only with this schema:
            {{
            "sentiment": "positive|neutral|negative"
            }}"""
        response = safe_json_dumps({"sentiment": label})
        return build_chat_example(prompt, response)

    train_ds = ds["train"].map(mapper, remove_columns=ds["train"].column_names)
    return train_ds


def convert_twitter_financial_news() -> Optional[Dataset]:
    """
    Example label mapping:
      0 -> bearish -> negative
      1 -> bullish -> positive
      2 -> neutral -> neutral

    Adjust if your version differs.
    """
    try:
        ds = load_dataset("zeroshot/twitter-financial-news-sentiment", trust_remote_code=True)
    except Exception as e:
        print(f"[WARN] Could not load Twitter financial sentiment dataset: {e}")
        return None

    label_map = {
        0: "negative",
        1: "positive",
        2: "neutral",
    }

    def mapper(ex):
        text = ex["text"]
        label = label_map[int(ex["label"])]
        prompt = f"""Task: classify_financial_sentiment
            Text:
            {text}

            Return JSON only with this schema:
            {{
            "sentiment": "positive|neutral|negative"
            }}"""
        response = safe_json_dumps({"sentiment": label})
        return build_chat_example(prompt, response)

    pieces = []
    for split in ds.keys():
        pieces.append(ds[split].map(mapper, remove_columns=ds[split].column_names))
    return concatenate_datasets(pieces)


def convert_fiqa() -> Optional[Dataset]:
    """
    FiQA variants differ. This tries to support common fields.
    We convert available QA-style rows into instruction-answer pairs.
    """
    candidate_names = [
        "LLukas22/fiqa",
        "pauri32/fiqa-2018",
    ]

    ds = None
    for name in candidate_names:
        try:
            ds = load_dataset(name)
            print(f"[INFO] Loaded FiQA from {name}")
            break
        except Exception:
            pass

    if ds is None:
        print("[WARN] Could not load FiQA")
        return None

    def mapper(ex):
        # Try common field names
        question = ex.get("question") or ex.get("title") or ex.get("input") or ""
        answer = ex.get("answer") or ex.get("gold_answer") or ex.get("output") or ""
        context = ex.get("context") or ex.get("sentence") or ex.get("text") or ""

        prompt = f"""Task: answer_financial_question
            Question:
            {question}

            Context:
            {context}

            Answer the question clearly."""
        response = str(answer).strip()
        if not response:
            response = "Insufficient information."

        return build_chat_example(prompt, response)

    pieces = []
    for split in ds.keys():
        pieces.append(ds[split].map(mapper, remove_columns=ds[split].column_names))
    return concatenate_datasets(pieces)


def convert_custom_entity_affect_jsonl(path: str) -> Optional[Dataset]:
    rows = read_jsonl(path)
    if not rows:
        print(f"[WARN] No custom entity-affect data found at {path}")
        return None

    converted = []
    for row in rows:
        article = row["article"]
        entities = row["entities"]

        # optionally force exact count if you store it
        k = row.get("k", len(entities))

        prompt = f"""Task: extract_entities_and_affect
            Extract exactly {k} finance-relevant entities from the article.

            Use only these entity types:
            {FINANCE_ENTITY_TYPES}

            For each entity, return:
            - text
            - type
            - sentiment
            - valence
            - arousal
            - evidence

            Return strict JSON only with this schema:
            {{
            "entities": [
                {{
                "text": "...",
                "type": "...",
                "sentiment": "positive|neutral|negative",
                "valence": 0.0,
                "arousal": 0.0,
                "evidence": "..."
                }}
            ]
            }}

            Article:
            {article}"""
        response = safe_json_dumps({"entities": entities})
        converted.append(build_chat_example(prompt, response))

    return Dataset.from_list(converted)


def load_all_training_data() -> DatasetDict:
    train_parts = []
    eval_parts = []

    # ---------- Existing public datasets ----------
    fpb = convert_financial_phrasebank()
    if fpb is not None:
        split = fpb.train_test_split(test_size=0.1, seed=SEED)
        train_parts.append(split["train"])
        eval_parts.append(split["test"])

    tw = convert_twitter_financial_news()
    if tw is not None:
        split = tw.train_test_split(test_size=0.1, seed=SEED)
        train_parts.append(split["train"])
        eval_parts.append(split["test"])

    fiqa = convert_fiqa()
    if fiqa is not None:
        split = fiqa.train_test_split(test_size=0.1, seed=SEED)
        train_parts.append(split["train"])
        eval_parts.append(split["test"])

    # ---------- Your custom entity-affect supervision ----------
    ent_train = convert_custom_entity_affect_jsonl(CUSTOM_ENTITY_AFFECT_JSONL)
    ent_val = convert_custom_entity_affect_jsonl(CUSTOM_ENTITY_AFFECT_VAL_JSONL)

    if ent_train is not None:
        train_parts.append(ent_train)
    if ent_val is not None:
        eval_parts.append(ent_val)

    if not train_parts:
        raise ValueError("No training datasets could be loaded.")

    train_ds = concatenate_datasets(train_parts).shuffle(seed=SEED)
    eval_ds = concatenate_datasets(eval_parts).shuffle(seed=SEED) if eval_parts else None

    if DEBUG_MAX_TRAIN is not None:
        train_ds = train_ds.select(range(min(DEBUG_MAX_TRAIN, len(train_ds))))
    if DEBUG_MAX_EVAL is not None and eval_ds is not None:
        eval_ds = eval_ds.select(range(min(DEBUG_MAX_EVAL, len(eval_ds))))

    dsdict = DatasetDict({"train": train_ds})
    if eval_ds is not None:
        dsdict["validation"] = eval_ds
    return dsdict

## Tokenization

In [6]:
def tokenize_dataset(dataset_dict: DatasetDict, tokenizer) -> DatasetDict:
    def tok_fn(example):
        text = format_messages_as_text(example["messages"], tokenizer)
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        # tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    tokenized = dataset_dict.map(
        tok_fn,
        remove_columns=dataset_dict["train"].column_names,
    )
    return tokenized

## Metrics

In [7]:
def compute_lm_metrics(eval_preds):
    """
    Simple perplexity-style reporting from eval loss will usually be enough
    for Trainer on causal LM. This is a placeholder if you want more later.
    """
    return {}

## Main Running

In [8]:
set_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------
# Load tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    #trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ----------------------------
# Load data
# ----------------------------
dataset_dict = load_all_training_data()
print(dataset_dict)
print(f"[INFO] Train size: {len(dataset_dict['train'])}")
if "validation" in dataset_dict:
    print(f"[INFO] Validation size: {len(dataset_dict['validation'])}")

tokenized = tokenize_dataset(dataset_dict, tokenizer)

# ----------------------------
# Load model
# ----------------------------
model_kwargs = {
    #"trust_remote_code": True,
    #"device_map": "auto",
    "torch_dtype": torch.float32,
    "low_cpu_mem_usage": True,
}

# if USE_4BIT:
#     from transformers import BitsAndBytesConfig

#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_quant_type="nf4",
#     )
#     model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_kwargs,
)

# if USE_4BIT:
#     model = prepare_model_for_kbit_training(model)

# ----------------------------
# LoRA config
# ----------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "up_proj",
        "down_proj",
        "gate_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ----------------------------
# Data collator
# ----------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# ----------------------------
# Training arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    eval_strategy="steps" if "validation" in tokenized else "no",
    # bf16=BF16,
    # fp16=FP16,
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=False,
    dataloader_num_workers=0,
    save_total_limit=2,
    gradient_checkpointing=False,
    use_cpu = True
)

# ----------------------------
# Trainer
# ----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"] if "validation" in tokenized else None,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_lm_metrics,
)

# ----------------------------
# Train
# ----------------------------
trainer.train()

# ----------------------------
# Save adapter + tokenizer
# ----------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"[INFO] Saved model to {OUTPUT_DIR}")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

c:\Program Files\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\great\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss,Validation Loss


[INFO] Saved model to ./qwen2.5_finance_lora


## Inference Section

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

article = """Apple rose after reporting stronger services revenue, while Treasury yields moved higher following hawkish comments from Fed officials."""

user_prompt = f"""Task: extract_entities_and_affect

    Extract exactly 5 finance-relevant entities from the article.

    Use only these entity types:
    {FINANCE_ENTITY_TYPES}

    For each entity, return:
    - text
    - type
    - sentiment
    - valence
    - arousal
    - evidence

    Return strict JSON only.

    Article:
    {article}
    """

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

text = tokenizer.decode(output[0], skip_special_tokens=True)
print(text)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\module.py:2586: UserWarning: for base_model.model.model.layers.13.self_attn.q_proj.lora_A.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
C:\Users\great\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\module.py:2586: UserWarning: for base_model.model.model.layers.13.self_attn.q_proj.lora_B.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copyin

KeyError: 'base_model.model.model.model.embed_tokens'